<a href="https://colab.research.google.com/github/CallMe-iO/Algoritma-Genetika-Pembuatan-jadwal/blob/main/template_Dev_Model_Deteksi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hubungkan dengan google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

Cek data

In [ ]:
import os
dataset_base_path = "/content/drive/MyDrive/LatihanYolo/DataPemain"

def count_images_in_dir(directory):
    count = 0
    if os.path.exists(directory):
        for f in os.listdir(directory):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                count += 1
    return count

print(f"Number of training images: {count_images_in_dir(os.path.join(dataset_base_path, 'images', 'train'))}")
print(f"Number of validation images: {count_images_in_dir(os.path.join(dataset_base_path, 'images', 'val'))}")
print(f"Number of test images: {count_images_in_dir(os.path.join(dataset_base_path, 'images', 'test'))}")

Coba ini, untuk mengambil (mengimpor) kelas YOLO dari pustaka ultralytics. Pustaka Ultralytics adalah library resmi untuk menjalankan model YOLO (You Only Look Once) seperti YOLOv5, YOLOv8, YOLOv9, hingga YOLOv11.

jika terjadi error:
jalankan ***!pip install ultralytics***
untuk Menginstal library ultralytics ke dalam lingkungan Python Anda.
Library ini adalah paket resmi untuk menjalankan berbagai model YOLO (YOLOv5 - YOLOv11).

In [ ]:
from ultralytics import YOLO

memuat model YOLO versi 11 (varian nano) ke dalam variabel bernama modelPemain.

varian lain yang bisa digunakan:
YOLO11n,
YOLO11s,
YOLO11m,
YOLO11l,
YOLO11x


In [ ]:
modelPemain = YOLO("yolo11n.pt")

perintah untuk melatih (training) model YOLO menggunakan dataset yang sudah didefinisikan dalam file sepakBola.yaml.

ambil alamat file sepakBola.yaml sesuai dengan alamat path file tersebut di gDrive



In [ ]:
modelPemain.train(data="/content/drive/MyDrive/LatihanYolo/sepakBola.yaml", epochs=10)

Evaluasi Model dengan Data Test

In [ ]:
model = YOLO('/content/runs/detect/train/weights/best.pt')
results = model.val(data="/content/drive/MyDrive/LatihanYolo/sepakBola.yaml", split='test')

precision = results.box.p
recall = results.box.r
map50 = results.box.map50

print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"mAP50: {map50}")

Lihat hasil prediksi pada suatu citra di folder test

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os
from IPython.display import Image, display

# Load the trained model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Define class names and their corresponding colors (BGR format for OpenCV)
# Based on the previous training output, class 0 is 'timMerah' and class 1 is 'timPutih'.
class_names = ['timMerah', 'timPutih']
class_colors = {
    'timMerah': (255, 0, 0),  # Red in BGR
    'timPutih': (255, 255, 255) # White in BGR
}

# Path to the test images
test_image_dir = '/content/drive/MyDrive/LatihanYolo/DataPemain_augmented/images/test'

# Get all image files in the test directory
image_files = [os.path.join(test_image_dir, f) for f in os.listdir(test_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Iterate through each image file and perform detection
for img_path in image_files:
    print(f"Processing image: {os.path.basename(img_path)}")
    # Perform prediction
    # conf: object confidence threshold for detection
    # iou: intersection over union (IoU) threshold for NMS
    results = model.predict(source=img_path, conf=0.25, iou=0.5, show=False, save=False, verbose=False)

    # Load the image for drawing
    img = cv2.imread(img_path)
    if img is None:
        print(f"Could not load image {img_path}. Skipping.")
        continue

    # Convert to RGB for display with IPython.display.Image
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Iterate through detections
    for r in results:
        boxes = r.boxes # Boxes object for bbox outputs
        for box in boxes:
            # Get bounding box coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            # Get class ID and confidence
            class_id = int(box.cls[0])
            conf = float(box.conf[0])

            # Get class name and color
            class_name = class_names[class_id]
            color = class_colors[class_name] # BGR color for OpenCV

            # Draw bounding box
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2) # Thickness 2

            # Put label
            label = f'{class_name} {conf:.2f}'
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.7
            font_thickness = 2
            text_size = cv2.getTextSize(label, font, font_scale, font_thickness)[0]
            text_x = x1
            text_y = y1 - 10 if y1 - 10 > text_size[1] else y1 + text_size[1] + 10

            # Draw text background rectangle
            cv2.rectangle(img_rgb, (text_x, text_y - text_size[1] - 5), (text_x + text_size[0] + 5, text_y + 5), color, -1)
            # Draw text (choose contrasting color for text, e.g., black if background is light, white if background is dark)
            text_color = (0, 0, 0) if np.mean(color) > 127 else (255, 255, 255)
            cv2.putText(img_rgb, label, (text_x + 2, text_y), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

    # Display the image with detections
    display(Image(data=cv2.imencode('.jpg', cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))[1].tobytes()))

jalankan pada video

In [ ]:
import cv2
from ultralytics import YOLO
import os

# Load the trained model
model = YOLO('/content/runs/detect/train2/weights/best.pt')

# Define class names and their corresponding colors (BGR format for OpenCV)
class_names = ['timMerah', 'timPutih']
class_colors = {
    'timMerah': (0, 0, 255),  # Red in BGR for class 0
    'timPutih': (255, 255, 255) # White in BGR for class 1
}

# Input video path
video_path = '/content/drive/MyDrive/LatihanYolo/video/test 29 detik WhatsApp Video 2025-12-03 at 19.05.09_3f04c641.mp4'

# Output video path
output_video_path = '/content/drive/MyDrive/LatihanYolo/video/output_detection_video test 29 detik WhatsApp Video 2025-12-03 at 19.05.09_3f04c641.mp4'

# Open the video file
cap = cv2.VideoCapture(video_path)

# Check if video opened successfully
if not cap.isOpened():
    print(f"Error: Could not open video file {video_path}")
else:
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4 files

    # Initialize VideoWriter to save the output video
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"Processing video: {os.path.basename(video_path)}")
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break # End of video

        frame_count += 1
        if frame_count % 30 == 0: # Process every 30th frame (roughly 1 frame per second for 30fps video)
            print(f"Processing frame {frame_count}...")

        # Perform prediction on the frame
        results = model.predict(source=frame, conf=0.25, iou=0.5, show=False, save=False, verbose=False)

        # Draw bounding boxes on the frame
        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                class_id = int(box.cls[0])

                # Get color based on class_id
                class_name = class_names[class_id]
                color = class_colors[class_name]

                # Draw bounding box without class name or confidence
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2) # Thickness 2

        # Write the processed frame to the output video
        out.write(frame)

    # Release resources
    cap.release()
    out.release()
    print(f"Video processing complete. Output saved to: {output_video_path}")

**Menginstal dua library:**

1.   albumentations

    Library untuk augmentasi data citra (rotasi, flip, blur, brightness, cropping, dsb).

    Sangat sering digunakan untuk training model deteksi/segmentasi.

2.   opencv-python-headless

    Versi OpenCV tanpa GUI.
    Digunakan di server/cloud seperti Google Colab agar tidak terjadi konflik dengan paket lain.
    Bisa membaca, menulis, dan memproses gambar.

In [ ]:
import sys
!{sys.executable} -m pip install albumentations opencv-python-headless

print("albumentations and opencv-python-headless installed.")

Proses Augmentasi

perhatikan folder dataset_base_path untuk data awal dan hasil augmentasi

perhatikan *num_augmentations_per_image* : berapa banyak augmentasi akan dilakukan

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# Define paths
dataset_base_path = '/content/drive/MyDrive/LatihanYolo/DataPemain'
train_images_dir = os.path.join(dataset_base_path, 'images', 'train')
train_labels_dir = os.path.join(dataset_base_path, 'labels', 'train')

# Create a directory for augmented data
augmented_images_dir = os.path.join(dataset_base_path, 'images', 'train')
augmented_labels_dir = os.path.join(dataset_base_path, 'labels', 'train')
os.makedirs(augmented_images_dir, exist_ok=True)
os.makedirs(augmented_labels_dir, exist_ok=True)

# Define the augmentation pipeline
# bbox_params tells albumentations how to handle bounding boxes
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Rotate(limit=30, p=0.7, border_mode=cv2.BORDER_CONSTANT),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.GaussNoise(p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.7),
    A.ISONoise(p=0.3),
    A.FancyPCA(alpha=0.1, p=0.3),
    A.RandomGamma(p=0.5),
    A.ColorJitter(p=0.5),
    A.ChannelShuffle(p=0.3),
    A.ToGray(p=0.2),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# --- Augment and save data ---
# Get a list of all image files in the training directory
image_files = [f for f in os.listdir(train_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

num_augmentations_per_image = 10 # Generate 10 augmented versions for each original image

print(f"Generating {num_augmentations_per_image} augmented versions for each of {len(image_files)} original images...")

for img_file in tqdm(image_files, desc="Augmenting Images"):
    img_name_without_ext = os.path.splitext(img_file)[0]
    image_path = os.path.join(train_images_dir, img_file)
    label_path = os.path.join(train_labels_dir, img_name_without_ext + '.txt')

    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Read bounding boxes and class labels from YOLO format
    bboxes = []
    class_labels = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = list(map(float, line.strip().split()))
                class_id = int(parts[0])
                # YOLO format: class_id x_center y_center width height
                # Albumentations expects (x_center, y_center, width, height, class_id)
                # Store class_id separately for label_fields
                bboxes.append(parts[1:])
                class_labels.append(class_id)

    if not bboxes:
        print(f"Warning: No bounding boxes found for {img_file}. Skipping augmentation for this image.")
        continue

    for i in range(num_augmentations_per_image):
        augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        aug_image = augmented['image']
        aug_bboxes = augmented['bboxes']
        aug_class_labels = augmented['class_labels']

        # Save augmented image
        aug_image_filename = f"{img_name_without_ext}_aug_{i}.png"
        aug_image_path = os.path.join(augmented_images_dir, aug_image_filename)
        cv2.imwrite(aug_image_path, cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR))

        # Save augmented labels
        aug_label_filename = f"{img_name_without_ext}_aug_{i}.txt"
        aug_label_path = os.path.join(augmented_labels_dir, aug_label_filename)
        with open(aug_label_path, 'w') as f:
            for bbox, label in zip(aug_bboxes, aug_class_labels):
                # Albumentations bbox format is (x_center, y_center, width, height)
                # We need to re-add class_id at the beginning for YOLO format
                f.write(f"{label} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}\n")

print("Data augmentation complete.")
print(f"Original training images: {len(image_files)}")
print(f"Augmented training images: {len(os.listdir(augmented_images_dir))}")
